In [13]:
import os
import re
import pandas as pd
import numpy as np

# Paths

topdir = "/Users/sm6511/Desktop/Prediction-Accomodation-Exp"

kathryn_path = os.path.join(
    topdir,
    "Analysis/validation/explanations/kathryn_data/kathryn_coding_train.xlsx"
)

my_coding_dir = os.path.join(
    topdir,
    "data/Combined/explanationcodes"
)

output_dir = os.path.join(
    topdir,
    "Analysis/validation/explanations/kathryn_data/comparison"
)

os.makedirs(output_dir, exist_ok=True)


# Helper functions

def clean_colname(x):
    """
    Standardize column names so capitalization/spacing differences
    don't cause mismatches.
    """
    if pd.isna(x):
        return ""

    x = str(x).strip().lower()

    # Standardize variants
    replacements = {
        "causal explanation": "causal_explanation",
        "causal_explanation": "causal_explanation",
        "none": "none",
        "participant": "participant",
        "shape": "shape",
        "tail": "tail",
        "color": "color",
        "wing": "wing",
        "wings": "wing",
        "stripes": "stripes",
        "feet": "feet",
        "all": "all",
    }

    return replacements.get(x, x.replace(" ", "_"))


def binary_code(x):
    """
    Treat only actual 1 / '1' as coded.
    Everything else, including blanks, backticks, etc., becomes 0.
    """
    if pd.isna(x):
        return 0

    if isinstance(x, (int, float, np.integer, np.floating)):
        return int(x == 1)

    return int(str(x).strip() == "1")


# Read Kathryn coding

raw = pd.read_excel(
    kathryn_path,
    header=None
)

print("Kathryn workbook shape:", raw.shape)


# Parse study sections

kathryn_studies = {}

study_start_rows = []

for i in range(len(raw)):

    first_cell = raw.iloc[i, 0]

    if pd.notna(first_cell):
        match = re.match(
            r"^\s*study\s*(\d+)\s*$",
            str(first_cell),
            flags=re.IGNORECASE
        )

        if match:
            study_num = int(match.group(1))
            study_start_rows.append((study_num, i))


print("\nDetected study sections:")
print(study_start_rows)


for idx, (study_num, start_row) in enumerate(study_start_rows):

    # The row containing "Study #" also contains the column headers
    header_row = raw.iloc[start_row].tolist()

    # Determine where this study section ends
    if idx < len(study_start_rows) - 1:
        end_row = study_start_rows[idx + 1][1]
    else:
        end_row = len(raw)

    # Data start on row immediately after header
    section = raw.iloc[start_row + 1:end_row].copy()

    # Use the Study row as header
    section.columns = [
        clean_colname(x)
        for x in header_row
    ]

    # First column is just the Study label column and should be ignored
    first_col = section.columns[0]

    if first_col.startswith("study"):
        section = section.drop(columns=first_col)

    # Remove columns with blank names
    section = section.loc[
        :,
        [c != "" for c in section.columns]
    ]

    # Remove completely empty rows
    section = section.dropna(how="all")

    # Participant must exist
    if "participant" not in section.columns:
        print(
            f"\nWARNING: Study {study_num} has no participant column."
        )
        continue

    # Drop rows without participant IDs
    section = section.dropna(
        subset=["participant"]
    ).copy()

    # Make participant IDs integers
    section["participant"] = pd.to_numeric(
        section["participant"],
        errors="coerce"
    )

    section = section.dropna(
        subset=["participant"]
    )

    section["participant"] = (
        section["participant"]
        .astype(int)
    )

    # Drop 'all' column (redundant)
    if "all" in section.columns:
        section = section.drop(columns="all")

    # Convert all coding variables to binary
    coding_cols = [
        col
        for col in section.columns
        if col != "participant"
    ]

    for col in coding_cols:
        section[col] = section[col].apply(binary_code)

    kathryn_studies[study_num] = section

    print(f"\nStudy {study_num} Kathryn columns:")
    print(section.columns.tolist())

    print(f"N = {len(section)}")


# Compare Kathryn's coding to mine

summary_rows = []
all_mismatches = []


for study_num, kathryn_df in kathryn_studies.items():

    print("\n" + "=" * 70)
    print(f"STUDY {study_num}")
    print("=" * 70)

    my_path = os.path.join(
        my_coding_dir,
        f"Explanation_Coding_Study{study_num}.csv"
    )

    if not os.path.exists(my_path):
        print(f"Could not find:")
        print(my_path)
        continue

   
    # Load my coding
   

    my_df = pd.read_csv(my_path)

    # Standardize column names
    my_df.columns = [
        clean_colname(c)
        for c in my_df.columns
    ]

    # Drop empty participant rows
    my_df = my_df.dropna(
        subset=["participant"]
    ).copy()

    my_df["participant"] = pd.to_numeric(
        my_df["participant"],
        errors="coerce"
    )

    my_df = my_df.dropna(
        subset=["participant"]
    )

    my_df["participant"] = (
        my_df["participant"]
        .astype(int)
    )

    # Drop redundant all column
    if "all" in my_df.columns:
        my_df = my_df.drop(columns="all")

    # Check columns

    kathryn_cols = set(kathryn_df.columns) - {"participant"}
    my_cols = set(my_df.columns) - {"participant"}

    common_cols = sorted(
        kathryn_cols.intersection(my_cols)
    )

    only_kathryn = sorted(
        kathryn_cols - my_cols
    )

    only_mine = sorted(
        my_cols - kathryn_cols
    )

    print("\nCommon coding columns:")
    print(common_cols)

    if only_kathryn:
        print("\nColumns only in Kathryn's coding:")
        print(only_kathryn)

    if only_mine:
        print("\nColumns only in my coding:")
        print(only_mine)

    # Convert common columns to binary

    for col in common_cols:
        my_df[col] = my_df[col].apply(binary_code)

    # Align by participant

    merged = kathryn_df[
        ["participant"] + common_cols
    ].merge(
        my_df[
            ["participant"] + common_cols
        ],
        on="participant",
        how="outer",
        suffixes=("_kathryn", "_mine"),
        indicator=True
    )

    print("\nParticipant matching:")
    print(merged["_merge"].value_counts())

    # Only participants coded by BOTH of us should enter
    # agreement 
    matched = merged[
        merged["_merge"] == "both"
    ].copy()

    # Compare each coding category

    for col in common_cols:

        kathryn_col = f"{col}_kathryn"
        my_col = f"{col}_mine"

        agree = (
            matched[kathryn_col] ==
            matched[my_col]
        )

        both_one = (
            (matched[kathryn_col] == 1) &
            (matched[my_col] == 1)
        )

        kathryn_only_one = (
            (matched[kathryn_col] == 1) &
            (matched[my_col] == 0)
        )

        mine_only_one = (
            (matched[kathryn_col] == 0) &
            (matched[my_col] == 1)
        )

        n = len(matched)

        summary_rows.append({
            "study": study_num,
            "category": col,
            "n_participants": n,
            "agreements": agree.sum(),
            "agreement_proportion": (
                agree.mean()
                if n > 0
                else np.nan
            ),
            "both_coded_1": both_one.sum(),
            "kathryn_1_mine_0": kathryn_only_one.sum(),
            "kathryn_0_mine_1": mine_only_one.sum()
        })

        # Save individual disagreements

        disagreements = matched[
            kathryn_only_one |
            mine_only_one
        ][
            [
                "participant",
                kathryn_col,
                my_col
            ]
        ].copy()

        if len(disagreements) > 0:

            disagreements["study"] = study_num
            disagreements["category"] = col

            disagreements = disagreements.rename(
                columns={
                    kathryn_col: "kathryn",
                    my_col: "mine"
                }
            )

            all_mismatches.append(
                disagreements[
                    [
                        "study",
                        "participant",
                        "category",
                        "kathryn",
                        "mine"
                    ]
                ]
            )


# Create summary

summary_df = pd.DataFrame(summary_rows)

print("\n\n")
print("=" * 70)
print("AGREEMENT SUMMARY")
print("=" * 70)

print(
    summary_df.to_string(index=False)
)


# Combine all mismatches

if all_mismatches:

    mismatch_df = pd.concat(
        all_mismatches,
        ignore_index=True
    )

    mismatch_df = mismatch_df.sort_values(
        [
            "study",
            "participant",
            "category"
        ]
    )

else:

    mismatch_df = pd.DataFrame(
        columns=[
            "study",
            "participant",
            "category",
            "kathryn",
            "mine"
        ]
    )


print("\n")
print("=" * 70)
print("DISAGREEMENTS")
print("=" * 70)

print(
    mismatch_df.to_string(index=False)
)


# Save results

'''
summary_df.to_csv(
    os.path.join(
        output_dir,
        "kathryn_coding_agreement_summary.csv"
    ),
    index=False
)

mismatch_df.to_csv(
    os.path.join(
        output_dir,
        "kathryn_coding_disagreements.csv"
    ),
    index=False
)
'''
print("\nSaved comparison files to:")
print(output_dir)

Kathryn workbook shape: (289, 9)

Detected study sections:
[(1, 0), (2, 16), (3, 38), (4, 61), (5, 83)]

Study 1 Kathryn columns:
['participant', 'causal_explanation', 'shape', 'tail', 'color', 'other']
N = 15

Study 2 Kathryn columns:
['participant', 'causal_explanation', 'color', 'tail', 'wing', 'other']
N = 21

Study 3 Kathryn columns:
['participant', 'causal_explanation', 'color', 'feet', 'both', 'other']
N = 22

Study 4 Kathryn columns:
['participant', 'causal_explanation', 'stripes', 'feet', 'both', 'other']
N = 21

Study 5 Kathryn columns:
['participant', 'causal_explanation', 'stripes', 'feet', 'both', 'other']
N = 30

STUDY 1

Common coding columns:
['causal_explanation', 'color', 'other', 'shape', 'tail']

Columns only in my coding:
['none']

Participant matching:
_merge
right_only    135
both           15
left_only       0
Name: count, dtype: int64

STUDY 2

Common coding columns:
['causal_explanation', 'color', 'other', 'tail', 'wing']

Columns only in my coding:
['none']



In [ ]:
alpha_rows = []

for study_num, kathryn_df in kathryn_studies.items():

    my_path = os.path.join(
        my_coding_dir,
        f"Explanation_Coding_Study{study_num}.csv"
    )

    if not os.path.exists(my_path):
        continue

    my_df = pd.read_csv(my_path)
    my_df.columns = [clean_colname(c) for c in my_df.columns]

    my_df["participant"] = pd.to_numeric(
        my_df["participant"],
        errors="coerce"
    )
    my_df = my_df.dropna(subset=["participant"])
    my_df["participant"] = my_df["participant"].astype(int)

    # Ignore redundant "all" column
    if "all" in my_df.columns:
        my_df = my_df.drop(columns="all")

    kathryn_cols = set(kathryn_df.columns) - {"participant"}
    my_cols = set(my_df.columns) - {"participant"}

    common_cols = sorted(kathryn_cols & my_cols)

    for col in common_cols:
        my_df[col] = my_df[col].apply(binary_code)

    merged = kathryn_df[
        ["participant"] + common_cols
    ].merge(
        my_df[["participant"] + common_cols],
        on="participant",
        how="inner",
        suffixes=("_kathryn", "_mine")
    )

    # Convert judgments to long form
    for col in common_cols:
        for _, row in merged.iterrows():
            alpha_rows.append({
                "study": study_num,
                "participant": row["participant"],
                "category": col,
                "kathryn": row[f"{col}_kathryn"],
                "mine": row[f"{col}_mine"]
            })


alpha_df = pd.DataFrame(alpha_rows)

In [12]:
print(alpha_df.head(50))

    study  participant            category  kathryn  mine
0       1            3  causal_explanation        0     0
1       1           22  causal_explanation        0     0
2       1           37  causal_explanation        0     0
3       1           42  causal_explanation        1     0
4       1           43  causal_explanation        0     0
5       1           48  causal_explanation        0     0
6       1           52  causal_explanation        0     0
7       1           61  causal_explanation        0     0
8       1           85  causal_explanation        0     0
9       1          115  causal_explanation        0     0
10      1          124  causal_explanation        0     0
11      1          130  causal_explanation        0     0
12      1          136  causal_explanation        0     0
13      1          142  causal_explanation        0     0
14      1          149  causal_explanation        1     1
15      1            3               color        0     0
16      1     

In [ ]:
def cronbach_alpha(df):
    """
    Rows = coding decisions/items
    Columns = raters
    """
    x = df.dropna().astype(float)

    print('data shape:', x.shape)
    k = x.shape[1] #Coding decisions x 2 coders (me and kathryn)

    item_variances = x.var(axis=0, ddof=1)
    print('item variances:')
    print(item_variances) #variance for each coder's coding decisions
    total_scores = x.sum(axis=1)
    print('total scores:')
    print(total_scores) #total score for each coding decision (sum of both coders' decisions)
    total_variance = total_scores.var(ddof=1)
    print('total variance:')
    print(total_variance) #variance of combined scores across all coding decisions

    if total_variance == 0:
        return np.nan

    alpha = (
        k / (k - 1)
    ) * (
        1 - item_variances.sum() / total_variance
    )

    return alpha

overall_alpha = cronbach_alpha(
    alpha_df[["kathryn", "mine"]]
)

print(f"Overall Cronbach's alpha: {overall_alpha:.3f}")
print("\nCronbach's alpha by category:")

for category, group in alpha_df.groupby("category"):

    alpha = cronbach_alpha(
        group[["kathryn", "mine"]]
    )

    print(
        f"{category}: alpha = {alpha:.3f}"
    )


data shape: (523, 2)
item variances:
kathryn    0.111910
mine       0.113331
dtype: float64
total scores:
0      0.0
1      0.0
2      0.0
3      1.0
4      0.0
      ... 
518    0.0
519    0.0
520    0.0
521    0.0
522    1.0
Length: 523, dtype: float64
total variance:
0.38726621392936966
Overall Cronbach's alpha: 0.837

Cronbach's alpha by category:
data shape: (51, 2)
item variances:
kathryn    0.073725
mine       0.090196
dtype: float64
total scores:
268    0.0
269    0.0
270    0.0
271    0.0
272    0.0
273    0.0
274    0.0
275    2.0
276    0.0
277    2.0
278    0.0
279    0.0
280    0.0
281    0.0
282    0.0
283    0.0
284    0.0
285    0.0
286    2.0
287    0.0
288    0.0
373    0.0
374    0.0
375    0.0
376    0.0
377    0.0
378    0.0
379    0.0
380    0.0
381    0.0
382    0.0
383    0.0
384    0.0
385    0.0
386    0.0
387    2.0
388    0.0
389    0.0
390    0.0
391    0.0
392    0.0
393    0.0
394    0.0
395    0.0
396    0.0
397    1.0
398    0.0
399    0.0
400    0.0
40

In [19]:
from sklearn.metrics import cohen_kappa_score

def cohens_kappa(df):
    """
    Rows = coding decisions
    Columns = raters
    """
    x = df.dropna()

    print("data shape:", x.shape)

    kappa = cohen_kappa_score(
        x["kathryn"],
        x["mine"]
    )

    return kappa


# Overall kappa
overall_kappa = cohens_kappa(
    alpha_df[["kathryn", "mine"]]
)

print(f"Overall Cohen's kappa: {overall_kappa:.3f}")


# Kappa by coding category
print("\nCohen's kappa by category:")

for category, group in alpha_df.groupby("category"):

    kappa = cohens_kappa(
        group[["kathryn", "mine"]]
    )

    print(
        f"{category}: kappa = {kappa:.3f}"
    )

data shape: (523, 2)
Overall Cohen's kappa: 0.719

Cohen's kappa by category:
data shape: (51, 2)
both: kappa = 0.878
data shape: (109, 2)
causal_explanation: kappa = 0.803
data shape: (58, 2)
color: kappa = 0.709
data shape: (73, 2)
feet: kappa = 0.511
data shape: (109, 2)
other: kappa = 0.000
data shape: (15, 2)
shape: kappa = 1.000
data shape: (51, 2)
stripes: kappa = 0.558
data shape: (36, 2)
tail: kappa = 0.625
data shape: (21, 2)
wing: kappa = 0.786
